In [6]:
import requests
import pandas as pd

In [4]:
API_KEY = "7100453b01da1115307de2336cae16cb"  # replace with your key

# test with The Matrix (tmdbId = 603)
url = f"https://api.themoviedb.org/3/movie/603?api_key={API_KEY}"
response = requests.get(url)
data = response.json()

print(data['title'])
print(data['overview'])
print(data['vote_average'])

The Matrix
Set in the 22nd century, The Matrix tells the story of a computer hacker who joins a group of underground insurgents fighting the vast and powerful computers who now rule the earth.
8.243


In [7]:
links = pd.read_csv('../data/raw/ml-25m/links.csv')
print(links.shape)
links.head()

(62423, 3)


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [8]:
import time

def get_movie_details(tmdb_id, api_key):
    url = f"https://api.themoviedb.org/3/movie/{int(tmdb_id)}?api_key={api_key}"
    response = requests.get(url)
    if response.status_code != 200:
        return None
    data = response.json()
    return {
        'tmdbId': tmdb_id,
        'overview': data.get('overview', ''),
        'poster_path': data.get('poster_path', ''),
        'vote_average': data.get('vote_average', 0),
        'release_date': data.get('release_date', ''),
        'runtime': data.get('runtime', 0)
    }

# get top 1000 most rated movies
import pandas as pd
ratings_25m = pd.read_csv('../data/processed/ratings_25m_clean.csv')
top_movies = (ratings_25m.groupby('movieId')
              .size()
              .sort_values(ascending=False)
              .head(1000)
              .reset_index()['movieId'])

links = pd.read_csv('../data/raw/ml-25m/links.csv')
top_links = links[links['movieId'].isin(top_movies)].dropna(subset=['tmdbId'])

print(f"Fetching details for {len(top_links)} movies...")

Fetching details for 1000 movies...


In [9]:
movie_details = []

for i, row in top_links.iterrows():
    details = get_movie_details(row['tmdbId'], API_KEY)
    if details:
        details['movieId'] = row['movieId']
        movie_details.append(details)
    
    # avoid hitting API rate limits
    if len(movie_details) % 50 == 0:
        print(f"Fetched {len(movie_details)} movies...")
        time.sleep(1)

details_df = pd.DataFrame(movie_details)
details_df.to_csv('../data/processed/tmdb_details.csv', index=False)
print(f"Done! Saved {len(details_df)} movies")

Fetched 50 movies...
Fetched 100 movies...
Fetched 150 movies...
Fetched 200 movies...
Fetched 250 movies...
Fetched 300 movies...
Fetched 350 movies...
Fetched 400 movies...
Fetched 450 movies...
Fetched 500 movies...
Fetched 550 movies...
Fetched 600 movies...
Fetched 650 movies...
Fetched 700 movies...
Fetched 750 movies...
Fetched 800 movies...
Fetched 850 movies...
Fetched 900 movies...
Fetched 950 movies...
Fetched 1000 movies...
Done! Saved 1000 movies


In [10]:
movies_25m = pd.read_csv('../data/raw/ml-25m/movies.csv')
enriched = movies_25m.merge(links, on='movieId').merge(details_df, on='movieId')

print(enriched.shape)
enriched[['title', 'overview', 'poster_path', 'vote_average']].head()

(1000, 11)


,title,overview,poster_path,vote_average
0,Toy Story (1995),"Led by Woody, Andy's toys live happily in his ...",/uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg,7.972
1,Jumanji (1995),When siblings Judy and Peter discover an encha...,/vgpXmVaVyUL7GGiDeiK1mKEKzcX.jpg,7.244
2,Grumpier Old Men (1995),A family wedding reignites the ancient feud be...,/1FSXpj5e8l4KH6nVFO5SPUeraOt.jpg,6.481
3,Father of the Bride Part II (1995),Just when George Banks has recovered from his ...,/rj4LBtwQ0uGrpBnCELr716Qo3mw.jpg,6.280
4,Heat (1995),Obsessive master thief Neil McCauley leads a t...,/umSVjVdbVwtx5ryCA2QXL44Durm.jpg,7.900


In [11]:
enriched.to_csv('../data/processed/movies_enriched.csv', index=False)
print("Saved!")

Saved!


In [12]:
pip install streamlit

  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached toml-0.10.2-py2.py3-none-any.whl.metadata (7.1 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached itsdangerous-2.2.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached watchdog-6.0.0-py3-none-win_amd64.whl.metadata (44 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached gitdb-4.0.12-py3-none-any.whl.metadata (1.2 kB)
  Using cached markupsafe-3.0.3-cp312-cp312-win_amd64.whl.metadata (2.8 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached rpds_py-0.30.0-cp312-cp312-win_amd64.whl.metadata (4.2 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
   ---------------------------------------- 0.0/9.2 MB ? eta -:--:--
   ----------------------- -


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
